# Model Goal Status Audit V1

Purpose: verify the current state of the active model goal before opening another training run. The goal requires retained models to pass walk-forward windows, keep walk-forward MaxDD below 20%, align returns with the current best model, and keep the converted SOXLBacktest qlib buy/sell policy as the reference baseline.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists())
gate_path = ROOT / 'runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json'
frontier_path = ROOT / 'runtime/model_selection/card/model_retention_frontier_v1/summary.json'
frontier_csv = ROOT / 'runtime/model_selection/card/model_retention_frontier_v1/retention_frontier.csv'
baseline_csv = ROOT / 'runtime/production/card/soxl_backtest_policy_reimplementation_v1/legacy_comparison_metrics.csv'
manifest_path = ROOT / 'dashboard_data/v2/manifest.json'

gate = json.loads(gate_path.read_text())['summary']
frontier = json.loads(frontier_path.read_text())['summary']
manifest = json.loads(manifest_path.read_text())
baseline = pd.read_csv(baseline_csv)
frontier_rows = pd.read_csv(frontier_csv)

pd.DataFrame([
    {
        'reference_profile': gate['reference_profile'],
        'reference_return_pct': gate['reference_return_pct'],
        'reference_max_drawdown_pct': gate['reference_max_drawdown_pct'],
        'current_best_return_pct': gate['current_best_return_pct'],
        'required_best_return_pct': gate['required_best_return_pct'],
        'max_walkforward_drawdown_pct': gate['max_walkforward_drawdown_pct'],
        'passing_profile_count': gate['passing_profile_count'],
        'profiles_scanned': gate['profiles_scanned'],
    }
])

,reference_profile,reference_return_pct,reference_max_drawdown_pct,current_best_return_pct,required_best_return_pct,max_walkforward_drawdown_pct,passing_profile_count,profiles_scanned
0,soxl_backtest_policy_reimplementation_v1,132.077978,13.554089,224.361539,213.143462,20.0,0,0


In [2]:
baseline[['window', 'start', 'end', 'new_return_pct', 'legacy_return_pct', 'return_delta_pct', 'new_max_drawdown_pct', 'legacy_max_drawdown_pct']]

,window,start,end,new_return_pct,legacy_return_pct,return_delta_pct,new_max_drawdown_pct,legacy_max_drawdown_pct
0,ytd_2026,2026-01-02,2026-06-09,132.077978,132.077978,0.000000e+00,13.554089,13.554089
1,june_2026,2026-06-01,2026-06-09,3.138960,3.138960,4.440892e-15,1.528209,1.528209


In [3]:
frontier_rows.head(8)[[
    'frontier_rank',
    'experiment',
    'primary_blocker',
    'next_action',
    'target_revision_ready',
    'wf_windows_present',
    'failed_wf_windows',
    'worst_wf_max_drawdown_pct',
    'relative_return_gap_to_target_pp',
]]

,frontier_rank,experiment,primary_blocker,next_action,target_revision_ready,wf_windows_present,failed_wf_windows,worst_wf_max_drawdown_pct,relative_return_gap_to_target_pp
0,1,cap10_wf2020_return_preserving_target_revision_v1,target_revision_ready,build_return_preserving_wf2020_bull_participat...,True,"wf_2020,wf_2022,wf_2023_2024",wf_2020,18.8,29.02
1,2,short_volshock_drawdown_drift_feature_audit_v1,missing_walkforward_evidence,screen drawdown-drift reward/checkpoint penalt...,False,NaN,NaN,NaN,NaN
2,3,sparse_release_drawdown_budget_guard_screen_v1,missing_walkforward_evidence,move_drawdown_control_into_checkpoint_objectiv...,False,NaN,NaN,NaN,NaN
3,4,sparse_release_drawdown_control_checkpoint_sco...,missing_walkforward_evidence,run_fold_local_drawdown_control_bc_or_checkpoi...,False,NaN,NaN,NaN,NaN
4,5,sparse_release_drawdown_control_target_handoff_v1,missing_walkforward_evidence,train_or_checkpoint_score_state_objective_draw...,False,NaN,NaN,NaN,NaN
5,6,state_head_fold_consensus_target_v1,missing_walkforward_evidence,discover_stable_rebound_labels_or_new_state_fa...,False,NaN,NaN,NaN,NaN
6,7,state_objective_alignment_screen_v1,missing_walkforward_evidence,train_or_checkpoint_score_state_objective_targets,False,NaN,NaN,NaN,NaN
7,8,state_objective_checkpoint_selection_v1,missing_walkforward_evidence,train_state_objective_checkpoint_or_bc,False,NaN,NaN,NaN,NaN


In [4]:
pd.DataFrame([
    {
        'available_profiles': len(manifest['available_profiles']),
        'runnable_profiles': len(manifest.get('profile_groups', {}).get('runnable', [])),
        'reference_profiles': ','.join(manifest.get('profile_groups', {}).get('reference', [])),
        'evidence_only_profiles': len(manifest.get('profile_groups', {}).get('evidence_only', [])),
        'safe_backtest_end': manifest['safe_backtest_end'],
        'latest_market_date': manifest['latest_market_date'],
        'recommended_next_source': frontier['recommended_next_source'],
        'recommended_primary_blocker': frontier['recommended_primary_blocker'],
    }
])

,available_profiles,runnable_profiles,reference_profiles,evidence_only_profiles,safe_backtest_end,latest_market_date,recommended_next_source,recommended_primary_blocker
0,5,0,soxl_backtest_policy_reimplementation_v1,4,2026-06-09,2026-06-10,cap10_wf2020_return_preserving_target_revision_v1,target_revision_ready


## Conclusion

- Current strict retention gate target is three walk-forward windows (`wf_2020`, `wf_2022`, `wf_2023_2024`), walk-forward MaxDD `< 20%`, and YTD return floor `213.143462%` versus the SOXLBacktest qlib baseline `132.077978%`.
- The SOXLBacktest qlib conversion remains aligned with legacy SOXLBacktest on YTD and June windows: YTD return delta `0.0pp`, YTD MaxDD `13.554089%`, and matched trade timing.
- No replayable RL model is currently promotable or runnable in dashboard v2: `passing_profile_count=0`, `profiles_scanned=0`, and manifest runnable profile count `0`.
- The next training path should start from `cap10_wf2020_return_preserving_target_revision_v1` with action `build_return_preserving_wf2020_bull_participation_target_then_rerun_three_fold_bc_checkpoint_metric_replay`; do not retain another executable actor until it passes the strict walk-forward gate.
- Frontend dashboard should continue presenting SOXLBacktest qlib as a reference baseline and mark other current models evidence-only until replay freshness exposes runnable profiles.